# Data Quality Assessment — Local Test Notebook

Two test modes:
1. **In-process** (Sections 2–3): runs the pipeline directly in Python — no server required.
2. **HTTP** (Section 4): sends `POST /data-quality/assess` to the running ml-server.

**Before running Section 4:** make sure Docker Compose is up (`docker compose up -d` in `ml-server/`).

---
## 0 — Configuration

In [1]:
import os, sys, json, pathlib
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Server ───────────────────────────────────────────────────────────────────
ML_SERVER_URL = os.getenv('ML_SERVER_BASE_URL', 'http://localhost:8030')

# ── SCADA source (used in HTTP mode only) ────────────────────────────────────
SCADA_URL  = os.getenv('SCADA_URL', 'http://127.0.0.1:7080/api/v1/read/archives')

# ── Archive path(s) to assess ────────────────────────────────────────────────
# Set to a real archive path, or leave as-is to use synthetic data.
OBJECT_REF = os.getenv('OBJECT_REF',
    '/root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value')

# ── Assessment window (empty string = auto-default) ──────────────────────────
FROM_TIME = os.getenv('DQ_FROM', '')   # ISO-8601 or Unix ms; '' → now-24h
TO_TIME   = os.getenv('DQ_TO',   '')   # ISO-8601 or Unix ms; '' → current hour
STEP      = int(os.getenv('DQ_STEP', '3600'))   # seconds

# ── Pipeline tuning ──────────────────────────────────────────────────────────
ALLOW_LOOK_AHEAD = True   # False → forward-fill only (production-safe mode)
Z_SCORE_WINDOW    = 48    # rolling window for spike detection
Z_SCORE_THRESHOLD = 3.0   # |Z| > threshold → spike
STUCK_WINDOW      = 10    # min consecutive identical values → stuck signal

# ── Add src/ to path (for in-process tests) ──────────────────────────────────
def _find_src():
    current = pathlib.Path(os.path.abspath('.'))
    for _ in range(8):
        cand = current / 'src'
        if (cand / 'api' / 'data_quality').is_dir():
            return str(cand)
        current = current.parent
    return None

_src = _find_src()
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

print(f'ML Server : {ML_SERVER_URL}')
print(f'SCADA URL : {SCADA_URL}')
print(f'Archive   : {OBJECT_REF}')
print(f'Step      : {STEP}s')
print(f'src/      : {_src or "NOT FOUND"}')

ML Server : http://localhost:8030
SCADA URL : http://127.0.0.1:7080/api/v1/read/archives
Archive   : /root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value
Step      : 3600s
src/      : /Users/rustamkrikbayev/Documents/projects/forecast/ml-server/src


---
## 1 — Health Check

In [2]:
def _check(label, fn):
    try:
        ok, detail = fn()
        print(f"{'✅' if ok else '❌'}  {label}: {detail}")
        return ok
    except Exception as e:
        print(f'❌  {label}: {type(e).__name__}: {e}')
        return False

_check('ml-server /ui/runtime-status', lambda: (
    (r := requests.get(f'{ML_SERVER_URL}/ui/runtime-status', timeout=5)).status_code == 200,
    f'HTTP {r.status_code}'
))
_check('ml-server /data-quality/assess (OPTIONS)', lambda: (
    (r := requests.options(f'{ML_SERVER_URL}/data-quality/assess', timeout=5))
    .status_code in (200, 405),   # 405 = route exists but OPTIONS not allowed
    f'HTTP {r.status_code}'
))

❌  ml-server /ui/runtime-status: ConnectionError: HTTPConnectionPool(host='localhost', port=8030): Max retries exceeded with url: /ui/runtime-status (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8030): Failed to establish a new connection: [Errno 61] Connection refused"))
❌  ml-server /data-quality/assess (OPTIONS): ConnectionError: HTTPConnectionPool(host='localhost', port=8030): Max retries exceeded with url: /data-quality/assess (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8030): Failed to establish a new connection: [Errno 61] Connection refused"))


False

---
## 2 — In-Process Pipeline Test with Synthetic Data

Builds a controlled time series with known defects (duplicates, gaps, spike, stuck signal),
runs the 4-step pipeline and checks that every anomaly is detected and correctly labelled.

In [ ]:
# ── Build synthetic series ────────────────────────────────────────────────────
# 168-point hourly window (7 days) with deliberate defects.

N_POINTS  = 168
STEP_S    = 3600
NOW_UTC   = datetime.now(tz=timezone.utc).replace(minute=0, second=0, microsecond=0)
END_UTC   = NOW_UTC
START_UTC = END_UTC - timedelta(seconds=(N_POINTS - 1) * STEP_S)

# Base signal: sine wave around 300 MW
rng    = np.random.default_rng(42)
t      = np.linspace(0, 4 * np.pi, N_POINTS)
signal = 300 + 60 * np.sin(t) + rng.normal(0, 5, N_POINTS)

grid_ms = np.array([
    int((START_UTC + timedelta(seconds=i * STEP_S)).timestamp() * 1000)
    for i in range(N_POINTS)
], dtype=np.int64)

# Inject defects
DEFECTS = {}

# 1. Duplicate timestamp at index 10
dup_ts = int(grid_ms[10])
DEFECTS['duplicate_idx'] = 10

# 2. Missing gap: remove indices 30–35 (6 points)
missing_idx = list(range(30, 36))
DEFECTS['missing_idx'] = missing_idx

# 3. Spike at index 50
spike_val = signal.copy()
spike_val[50] = 50_000.0
DEFECTS['spike_idx'] = 50

# 4. Stuck signal at indices 80–95 (16 identical values)
spike_val[80:96] = 300.0
DEFECTS['stuck_start'] = 80
DEFECTS['stuck_end']   = 95

# 5. One hard-limit (inf) at index 120
spike_val[120] = float('inf')
DEFECTS['inf_idx'] = 120

# Build raw SCADA payload (list of [ts_ms, value])
raw_series = [[int(ts), float(v)] for ts, v in zip(grid_ms, spike_val)]

# Add a duplicate entry for index 10
raw_series.insert(11, [dup_ts, float(spike_val[10]) + 20.0])   # second reading

# Remove the missing-gap indices from the raw payload
raw_series = [pt for i, pt in enumerate(raw_series)
              if pt[0] not in [int(grid_ms[j]) for j in missing_idx]]

print(f'Synthetic series: {len(raw_series)} raw points  (expected on grid: {N_POINTS})')
print(f'Window : {START_UTC.strftime("%Y-%m-%d %H:%M UTC")} → {END_UTC.strftime("%Y-%m-%d %H:%M UTC")}')
print(f'Injected defects:')
for k, v in DEFECTS.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Run pipeline in-process ───────────────────────────────────────────────────
from api.data_quality.models import AssessRequest, ScoringWeights
from api.data_quality.pipeline import DataQualityPipeline

req_synth = AssessRequest(
    **{'from': START_UTC.strftime('%Y-%m-%dT%H:%M:%SZ')},
    object_ref='SYNTHETIC_TAG',
    to=END_UTC.strftime('%Y-%m-%dT%H:%M:%SZ'),
    step=STEP_S,
    allow_look_ahead=ALLOW_LOOK_AHEAD,
    z_score_window=Z_SCORE_WINDOW,
    z_score_threshold=Z_SCORE_THRESHOLD,
    stuck_window=STUCK_WINDOW,
)

pipeline = DataQualityPipeline(req_synth)
result   = pipeline.run({'SYNTHETIC_TAG': raw_series})

stats = result.metrics_scoring.tags['SYNTHETIC_TAG']
print(f'Quality score  : {result.metrics_scoring.overall_quality_score:.1f} / 100')
print(f'Expected points: {stats.total_expected_points}')
print(f'Missing        : {stats.missing_points_count}')
print(f'Duplicates     : {stats.duplicates_count}')
print(f'Spike outliers : {stats.outliers_count}')
print(f'Stuck sequences: {stats.stuck_sequences_count}')
print(f'Rate-of-change : {stats.rate_of_change_count}')
print(f'Long gaps      : {stats.long_gaps_count}')
print(f'Anomaly records: {len(result.anomalies_log)}')

# ── Quick sanity checks ───────────────────────────────────────────────────────
def _check_assert(label, cond):
    print(f"  {'✅' if cond else '❌'}  {label}")

print('\nSanity checks:')
_check_assert('missing points detected',  stats.missing_points_count >= len(missing_idx))
_check_assert('duplicate detected',       stats.duplicates_count >= 1)
_check_assert('spike detected',           stats.outliers_count >= 1)
_check_assert('stuck signal detected',    stats.stuck_sequences_count >= 1)
_check_assert('score < 100',              result.metrics_scoring.overall_quality_score < 100)
_check_assert('cleaned_data non-empty',   len(result.cleaned_data) > 0)

In [ ]:
# ── Visualise: raw vs cleaned + anomaly markers ───────────────────────────────

# Parse cleaned series
df_cleaned = pd.DataFrame(result.cleaned_data)
df_cleaned['ts'] = pd.to_datetime(df_cleaned['timestamp'], utc=True).dt.tz_convert(None)
df_cleaned['value'] = df_cleaned['SYNTHETIC_TAG'].astype(float)

# Parse raw into a DataFrame
df_raw = pd.DataFrame(raw_series, columns=['ts_ms', 'value_raw'])
df_raw['ts'] = pd.to_datetime(df_raw['ts_ms'], unit='ms', utc=True).dt.tz_convert(None)
df_raw['value_raw'] = df_raw['value_raw'].replace([float('inf'), float('-inf')], float('nan'))

# Anomaly DataFrames by type
df_anom = pd.DataFrame([a.model_dump() for a in result.anomalies_log])
df_anom['ts'] = pd.to_datetime(df_anom['timestamp'], utc=True).dt.tz_convert(None)

ANOM_COLORS = {
    'missing':          '#3498db',
    'duplicate':        '#9b59b6',
    'hard_limit':       '#e74c3c',
    'spike_outlier':    '#e67e22',
    'stuck_signal':     '#f39c12',
    'rate_of_change':   '#1abc9c',
}

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('Raw input vs Cleaned output', 'Anomaly types over time'),
    row_heights=[0.65, 0.35],
    vertical_spacing=0.08,
)

# Row 1: raw and cleaned series
fig.add_trace(go.Scatter(
    x=df_raw['ts'], y=df_raw['value_raw'],
    mode='lines', name='Raw', opacity=0.5,
    line=dict(color='#95a5a6', width=1),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_cleaned['ts'], y=df_cleaned['value'],
    mode='lines', name='Cleaned',
    line=dict(color='#2ecc71', width=2),
), row=1, col=1)

# Row 1: anomaly markers
for atype, color in ANOM_COLORS.items():
    sub = df_anom[df_anom['anomaly_type'] == atype]
    if sub.empty:
        continue
    # Join with cleaned to get y position for markers
    sub_merged = sub.merge(df_cleaned[['ts', 'value']], on='ts', how='left')
    fig.add_trace(go.Scatter(
        x=sub_merged['ts'], y=sub_merged['value'],
        mode='markers', name=atype,
        marker=dict(color=color, size=8, symbol='x'),
        hovertemplate=f'<b>{atype}</b><br>%{{x}}<extra></extra>',
    ), row=1, col=1)

# Row 2: anomaly type categorical strip
atype_vals = {'missing': 1, 'duplicate': 2, 'hard_limit': 3,
              'spike_outlier': 4, 'stuck_signal': 5, 'rate_of_change': 6}
for atype, yval in atype_vals.items():
    sub = df_anom[df_anom['anomaly_type'] == atype]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['ts'], y=[yval] * len(sub),
        mode='markers', name=atype, showlegend=False,
        marker=dict(color=ANOM_COLORS[atype], size=6, symbol='circle'),
    ), row=2, col=1)

fig.update_yaxes(tickvals=list(atype_vals.values()),
                 ticktext=list(atype_vals.keys()), row=2, col=1)
fig.update_layout(
    title=f'Data Quality Pipeline — Synthetic Test  |  Score: {result.metrics_scoring.overall_quality_score:.1f}/100',
    hovermode='x unified', template='plotly_white', height=700,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

In [ ]:
# ── Anomaly log summary ───────────────────────────────────────────────────────
from IPython.display import display

summary = (
    df_anom
    .groupby(['anomaly_type', 'action_taken'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
print('Anomaly log summary:')
display(summary)

print(f'\nFirst 10 anomaly records:')
display(
    df_anom[['ts', 'anomaly_type', 'action_taken', 'raw_value', 'duration_seconds']]
    .sort_values('ts')
    .head(10)
    .reset_index(drop=True)
)

---
## 3 — In-Process: Parse-Time Variants

Verifies that `from` / `to` accept ISO strings, Unix ms timestamps, and empty strings.

In [ ]:
from api.data_quality.models import _parse_time

cases = [
    ('ISO string',          '2026-05-01T00:00:00Z'),
    ('ISO with offset',     '2026-05-01T05:00:00+05:00'),
    ('Unix ms (int)',        1746057600000),
    ('Unix ms (str)',        '1746057600000'),
    ('Empty string',        ''),
    ('None',                None),
]

print(f'{"Input":<30} {"Result":<35} {"Status"}')
print('-' * 75)
for label, value in cases:
    try:
        dt = _parse_time(value)
        result_str = dt.strftime('%Y-%m-%dT%H:%M:%SZ') if dt else 'None (→ default)'
        status = '✅'
    except Exception as e:
        result_str = f'ERROR: {e}'
        status = '❌'
    print(f'{label:<30} {result_str:<35} {status}')

# Test empty from/to defaults
print('\nEmpty from/to → auto-default:')
req_empty = AssessRequest(**{'from': ''}, object_ref='T', to='', step=3600)
print(f'  start_time : {req_empty.start_time.strftime("%Y-%m-%dT%H:%M:%SZ")}')
print(f'  end_time   : {req_empty.end_time.strftime("%Y-%m-%dT%H:%M:%SZ")}')
print(f'  window     : {(req_empty.end_time - req_empty.start_time).total_seconds() / 3600:.0f} h')

---
## 4 — HTTP Tests: POST /data-quality/assess

These cells call the running ml-server. Make sure `docker compose up -d` is running.

In [ ]:
# ── 4.1 Minimal request (empty from / to → server applies defaults) ───────────
endpoint = f'{ML_SERVER_URL}/data-quality/assess'

body_minimal = {
    'object_ref': OBJECT_REF,
    'from':       '',
    'to':         '',
    'step':       STEP,
}

print(f'POST {endpoint}')
print('Body:', json.dumps(body_minimal, indent=2))

resp = requests.post(endpoint, json=body_minimal, timeout=60)
HTTP_RESULT_MINIMAL = resp.json() if resp.content else {}

print(f'\nHTTP {resp.status_code}  ({resp.elapsed.total_seconds():.2f}s)')

if resp.status_code == 200:
    md  = HTTP_RESULT_MINIMAL.get('metadata', {})
    sc  = HTTP_RESULT_MINIMAL.get('metrics_scoring', {})
    print(f'  Window  : {md.get("timestamp_start")} → {md.get("timestamp_end")}')
    print(f'  Tags    : {md.get("total_tags_processed")}')
    print(f'  Score   : {sc.get("overall_quality_score")} / 100')
    print(f'  Elapsed : {md.get("elapsed_seconds")}s')
    print(f'  Anomalies: {len(HTTP_RESULT_MINIMAL.get("anomalies_log", []))}')
elif resp.status_code == 503:
    print(f'  503 — SCADA unavailable. Enable stub: SCADA_STUB_ENABLED=true')
    print(json.dumps(HTTP_RESULT_MINIMAL, indent=2))
else:
    print(json.dumps(HTTP_RESULT_MINIMAL, indent=2, ensure_ascii=False))

In [ ]:
# ── 4.2 Full request: explicit ISO window ─────────────────────────────────────
now_h = datetime.now(tz=timezone.utc).replace(minute=0, second=0, microsecond=0)
win_from = (now_h - timedelta(hours=168)).strftime('%Y-%m-%dT%H:%M:%SZ')  # 7 days
win_to   = now_h.strftime('%Y-%m-%dT%H:%M:%SZ')

body_full = {
    'object_ref':         OBJECT_REF,
    'from':               win_from,
    'to':                 win_to,
    'step':               STEP,
    'allow_look_ahead':   ALLOW_LOOK_AHEAD,
    'z_score_window':     Z_SCORE_WINDOW,
    'z_score_threshold':  Z_SCORE_THRESHOLD,
    'stuck_window':       STUCK_WINDOW,
    'weights': {
        'w_missing':   1.0,
        'w_duplicate': 0.2,
        'w_outlier':   0.8,
        'w_stuck':     1.0,
    },
}

print(f'POST {endpoint}')
print('Body:', json.dumps(body_full, indent=2))

resp_full = requests.post(endpoint, json=body_full, timeout=60)
HTTP_RESULT_FULL = resp_full.json() if resp_full.content else {}

print(f'\nHTTP {resp_full.status_code}  ({resp_full.elapsed.total_seconds():.2f}s)')

if resp_full.status_code == 200:
    md  = HTTP_RESULT_FULL.get('metadata', {})
    sc  = HTTP_RESULT_FULL.get('metrics_scoring', {})
    tags_sc = sc.get('tags', {})
    print(f'  Window  : {md.get("timestamp_start")} → {md.get("timestamp_end")}')
    print(f'  Score   : {sc.get("overall_quality_score")} / 100')
    for tag_id, tag_stats in tags_sc.items():
        short = tag_id.split('/')[-1] if '/' in tag_id else tag_id
        print(f'  {short}: {tag_stats["quality_score"]:.1f}/100  '
              f'miss={tag_stats["missing_points_count"]}  '
              f'spikes={tag_stats["outliers_count"]}  '
              f'stuck={tag_stats["stuck_sequences_count"]}')
else:
    print(json.dumps(HTTP_RESULT_FULL, indent=2, ensure_ascii=False))

In [ ]:
# ── 4.3 Unix millisecond timestamps ──────────────────────────────────────────
from_ms = int((now_h - timedelta(hours=24)).timestamp() * 1000)
to_ms   = int(now_h.timestamp() * 1000)

body_ms = {
    'object_ref': OBJECT_REF,
    'from':       from_ms,
    'to':         to_ms,
    'step':       STEP,
}

print(f'POST {endpoint}  (Unix ms timestamps)')
print('Body:', json.dumps(body_ms, indent=2))

resp_ms = requests.post(endpoint, json=body_ms, timeout=30)
print(f'\nHTTP {resp_ms.status_code}  ({resp_ms.elapsed.total_seconds():.2f}s)')

if resp_ms.status_code == 200:
    _r = resp_ms.json()
    _md = _r.get('metadata', {})
    print(f'  Parsed window : {_md.get("timestamp_start")} → {_md.get("timestamp_end")}')
    print(f'  Score         : {_r["metrics_scoring"]["overall_quality_score"]} / 100')
    print('  ✅  Unix ms timestamps parsed correctly')
else:
    print(resp_ms.text[:400])

---
## 5 — Visualise HTTP Result

In [ ]:
# ── 5.1 Quality score gauge + per-tag bar ─────────────────────────────────────
HTTP_RESULT = HTTP_RESULT_FULL if resp_full.status_code == 200 else HTTP_RESULT_MINIMAL

if HTTP_RESULT.get('metrics_scoring'):
    sc     = HTTP_RESULT['metrics_scoring']
    overall = sc['overall_quality_score']
    tags_sc = sc.get('tags', {})

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Overall Score', 'Per-Tag Breakdown'),
        specs=[[{'type': 'indicator'}, {'type': 'bar'}]],
    )

    # Gauge
    fig.add_trace(go.Indicator(
        mode='gauge+number+delta',
        value=overall,
        delta={'reference': 95.0, 'increasing': {'color': '#2ecc71'}},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': '#2ecc71' if overall >= 80 else '#e74c3c'},
            'steps': [
                {'range': [0,  60], 'color': '#fadbd8'},
                {'range': [60, 80], 'color': '#fdebd0'},
                {'range': [80, 100], 'color': '#d5f5e3'},
            ],
            'threshold': {
                'line': {'color': 'red', 'width': 3},
                'thickness': 0.75, 'value': 80,
            },
        },
        title={'text': 'Quality Score'},
        number={'suffix': ' / 100'},
    ), row=1, col=1)

    # Per-tag bar chart
    if tags_sc:
        tag_labels = [t.split('/')[-1] if '/' in t else t for t in tags_sc]
        tag_scores = [v['quality_score'] for v in tags_sc.values()]
        bar_colors = ['#2ecc71' if s >= 80 else '#e67e22' if s >= 60 else '#e74c3c'
                      for s in tag_scores]
        fig.add_trace(go.Bar(
            x=tag_labels, y=tag_scores,
            marker_color=bar_colors,
            text=[f'{s:.1f}' for s in tag_scores],
            textposition='outside',
            showlegend=False,
        ), row=1, col=2)
        fig.update_yaxes(range=[0, 110], row=1, col=2)

    fig.update_layout(
        height=400, template='plotly_white',
        title='Data Quality Assessment Results',
    )
    fig.show()
else:
    print('⚠  No HTTP result to visualise — run Section 4 first.')

In [ ]:
# ── 5.2 Per-tag defect breakdown stacked bar ───────────────────────────────────
if HTTP_RESULT.get('metrics_scoring'):
    tags_sc = HTTP_RESULT['metrics_scoring'].get('tags', {})

    records = []
    for tag_id, s in tags_sc.items():
        label = tag_id.split('/')[-1] if '/' in tag_id else tag_id
        records.append({
            'tag':     label,
            'Missing': s['missing_points_count'],
            'Duplicates': s['duplicates_count'],
            'Spikes':  s['outliers_count'],
            'Stuck':   s['stuck_sequences_count'],
            'RoC':     s['rate_of_change_count'],
        })

    df_stats = pd.DataFrame(records).set_index('tag')

    if not df_stats.empty and df_stats.sum().sum() > 0:
        fig2 = go.Figure()
        defect_colors = {
            'Missing':    '#3498db',
            'Duplicates': '#9b59b6',
            'Spikes':     '#e67e22',
            'Stuck':      '#f39c12',
            'RoC':        '#1abc9c',
        }
        for col, color in defect_colors.items():
            fig2.add_trace(go.Bar(
                name=col, x=df_stats.index.tolist(), y=df_stats[col].tolist(),
                marker_color=color,
            ))
        fig2.update_layout(
            barmode='stack', title='Defects per Tag',
            xaxis_title='Tag', yaxis_title='Point count',
            template='plotly_white', height=360,
        )
        fig2.show()
    else:
        print('No defects detected — data is clean!')

    display(df_stats)
else:
    print('⚠  No HTTP result — run Section 4 first.')

In [ ]:
# ── 5.3 Cleaned time series from HTTP result ───────────────────────────────────
if HTTP_RESULT.get('cleaned_data'):
    df_http = pd.DataFrame(HTTP_RESULT['cleaned_data'])
    df_http['ts'] = pd.to_datetime(df_http['timestamp'], utc=True).dt.tz_convert(None)

    tags_sc = HTTP_RESULT['metrics_scoring'].get('tags', {})
    tag_ids_http = list(tags_sc.keys())

    fig3 = go.Figure()
    colors_pal = ['#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']

    for i, tag_id in enumerate(tag_ids_http):
        if tag_id not in df_http.columns:
            continue
        label = tag_id.split('/')[-1] if '/' in tag_id else tag_id
        score = tags_sc[tag_id]['quality_score']
        fig3.add_trace(go.Scatter(
            x=df_http['ts'], y=df_http[tag_id].astype(float),
            mode='lines',
            name=f'{label}  ({score:.1f}/100)',
            line=dict(color=colors_pal[i % len(colors_pal)], width=1.8),
        ))

    # Anomaly markers from HTTP response
    if HTTP_RESULT.get('anomalies_log'):
        df_anom_http = pd.DataFrame(HTTP_RESULT['anomalies_log'])
        df_anom_http['ts'] = pd.to_datetime(
            df_anom_http['timestamp'], utc=True).dt.tz_convert(None)
        for atype in df_anom_http['anomaly_type'].unique():
            if atype in ('missing',):   # skip fill markers — too many
                continue
            sub = df_anom_http[df_anom_http['anomaly_type'] == atype]
            fig3.add_trace(go.Scatter(
                x=sub['ts'],
                y=[None] * len(sub),   # placed at top via annotation, not y-value
                mode='markers',
                name=atype,
                marker=dict(color=ANOM_COLORS.get(atype, '#888'), size=10, symbol='x'),
                hovertemplate=f'<b>{atype}</b><br>%{{x}}<extra></extra>',
                yaxis='y2',
            ))

    md_http = HTTP_RESULT.get('metadata', {})
    fig3.update_layout(
        title=f'Cleaned Data  |  Score: {HTTP_RESULT["metrics_scoring"]["overall_quality_score"]:.1f}/100  |  '
              f'{md_http.get("timestamp_start", "")} → {md_http.get("timestamp_end", "")}',
        xaxis_title='Time (UTC)', yaxis_title='Value',
        hovermode='x unified', template='plotly_white', height=450,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig3.show()
else:
    print('⚠  No cleaned_data in HTTP result — run Section 4 first.')

---
## 6 — Export Results

In [ ]:
# ── Export cleaned_data to CSV and anomaly_log to CSV ────────────────────────
import pathlib

EXPORT_SOURCE = HTTP_RESULT if HTTP_RESULT.get('cleaned_data') else result.model_dump()
export_dir    = pathlib.Path('.') / 'exports'
export_dir.mkdir(exist_ok=True)

ts_now   = datetime.now().strftime('%Y%m%d_%H%M%S')
tag_slug = OBJECT_REF.split('/')[-1][:40].replace(' ', '_')

# Cleaned data
if EXPORT_SOURCE.get('cleaned_data'):
    df_exp = pd.DataFrame(EXPORT_SOURCE['cleaned_data'])
    fpath_cleaned = export_dir / f'data_quality_cleaned__{tag_slug}__{ts_now}.csv'
    df_exp.to_csv(fpath_cleaned, index=False)
    print(f'✅  Cleaned data  → {fpath_cleaned}  ({len(df_exp)} rows)')

# Anomaly log
if EXPORT_SOURCE.get('anomalies_log'):
    df_anom_exp = pd.DataFrame(EXPORT_SOURCE['anomalies_log'])
    fpath_anom = export_dir / f'data_quality_anomalies__{tag_slug}__{ts_now}.csv'
    df_anom_exp.to_csv(fpath_anom, index=False)
    print(f'✅  Anomaly log   → {fpath_anom}  ({len(df_anom_exp)} records)')

# Full JSON report
fpath_json = export_dir / f'data_quality_report__{tag_slug}__{ts_now}.json'
with open(fpath_json, 'w') as f:
    json.dump(EXPORT_SOURCE, f, indent=2, ensure_ascii=False, default=str)
print(f'✅  Full report   → {fpath_json}')

---
## 7 — Batch Assessment from `inputs.csv`

Reads `local/models/training_workspace/models_enabled/inputs.csv` and `models.csv`,
joins them on `object_ref`, then runs the data quality pipeline for every input archive.

| Column | Source | Meaning |
|--------|--------|---------|
| `input_ref` | inputs.csv | SCADA archive path to assess |
| `api_url` | inputs.csv | SCADA endpoint for that archive |
| `object_ref` | inputs.csv | Model reference (join key) |
| `step` | models.csv | Time step in seconds |
| `input_range` | models.csv | History window in steps |

**No server needed** — fetches SCADA directly and runs the pipeline in-process.

In [ ]:
# ── Locate inputs.csv / models.csv ───────────────────────────────────────────
# Searches upward from the notebook directory, then looks for
# ../local/models/training_workspace/models_enabled/ relative to ml-server/

def _find_enabled_dir():
    current = pathlib.Path(os.path.abspath('.'))
    for _ in range(10):
        # Sibling 'local' next to ml-server
        candidate = current / 'local' / 'models' / 'training_workspace' / 'models_enabled'
        if candidate.is_dir():
            return candidate
        # Direct subdirectory
        candidate2 = current / 'models' / 'training_workspace' / 'models_enabled'
        if candidate2.is_dir():
            return candidate2
        current = current.parent
    return None

ENABLED_DIR = _find_enabled_dir()
if ENABLED_DIR is None:
    raise FileNotFoundError(
        'Cannot locate models_enabled/ directory. '
        'Set ENABLED_DIR manually below.')

# ── Override path here if needed ─────────────────────────────────────────────
# ENABLED_DIR = pathlib.Path('/absolute/path/to/models_enabled')

INPUTS_CSV = ENABLED_DIR / 'inputs.csv'
MODELS_CSV = ENABLED_DIR / 'models.csv'

# ── Assessment window ─────────────────────────────────────────────────────────
# BATCH_HOURS overrides input_range from models.csv (set to None to use models.csv value)
BATCH_HOURS = None   # e.g. 168 for 7 days; None → use input_range * step

# SCADA request timeout per archive
BATCH_SCADA_TIMEOUT = 30   # seconds

print(f'inputs.csv : {INPUTS_CSV}  (exists={INPUTS_CSV.exists()})')
print(f'models.csv : {MODELS_CSV}  (exists={MODELS_CSV.exists()})')

In [ ]:
# ── Load and join CSVs ────────────────────────────────────────────────────────

df_inputs = pd.read_csv(INPUTS_CSV, sep=';', comment='#',
                         names=['row_id', 'object_ref', 'pattern', 'input_ref', 'api_url'],
                         dtype=str).dropna(subset=['input_ref'])

df_models = pd.read_csv(MODELS_CSV, sep=';', comment='#',
                         names=['row_id', 'object_ref', 'input_range', 'output_range', 'step'],
                         dtype=str).dropna(subset=['object_ref'])
df_models['step']        = pd.to_numeric(df_models['step'],        errors='coerce').fillna(3600).astype(int)
df_models['input_range'] = pd.to_numeric(df_models['input_range'], errors='coerce').fillna(168).astype(int)

df_batch = df_inputs.merge(
    df_models[['object_ref', 'step', 'input_range']],
    on='object_ref', how='left'
)
df_batch['step']        = df_batch['step'].fillna(3600).astype(int)
df_batch['input_range'] = df_batch['input_range'].fillna(168).astype(int)
df_batch = df_batch.reset_index(drop=True)

print(f'Loaded {len(df_batch)} input(s) to assess:')
display(
    df_batch[['row_id', 'object_ref', 'input_ref', 'step', 'input_range', 'api_url']]
    .style.set_properties(**{'text-align': 'left'})
)

In [ ]:
# ── Batch assessment loop ─────────────────────────────────────────────────────
from urllib.parse import urlparse, urlunparse

from api.data_quality.models import AssessRequest
from api.data_quality.pipeline import DataQualityPipeline


def _normalize_scada_url(url: str) -> str:
    """Rewrite 127.0.0.1/localhost → host.docker.internal when inside Docker."""
    if not url:
        return url
    parsed = urlparse(url)
    if parsed.hostname not in ('127.0.0.1', 'localhost'):
        return url
    if not pathlib.Path('/.dockerenv').exists():
        return url
    netloc = parsed.netloc.replace(parsed.hostname, 'host.docker.internal')
    return urlunparse(parsed._replace(netloc=netloc))


def _fetch_scada(api_url: str, archive: str, from_ms: int, to_ms: int,
                 step_s: int, timeout: int = 30):
    """POST to SCADA and return raw payload dict, or None on error."""
    url = _normalize_scada_url(api_url)
    body = {'from': from_ms, 'to': to_ms, 'archive': [archive], 'step': step_s}
    try:
        resp = requests.post(url, json=body, timeout=timeout)
        if resp.status_code == 200:
            data = resp.json()
            # Expect {archive_path: [[ts, val], ...], ...}
            if isinstance(data, dict) and data:
                return data
        return None
    except Exception:
        return None


now_utc = datetime.now(tz=timezone.utc).replace(minute=0, second=0, microsecond=0)

BATCH_RESULTS = []   # list of dicts, one per input row

for _, row in df_batch.iterrows():
    object_ref  = str(row['object_ref']).strip()
    input_ref   = str(row['input_ref']).strip()
    api_url     = str(row['api_url']).strip()
    step_s      = int(row['step'])
    input_range = int(row['input_range'])

    window_h    = BATCH_HOURS if BATCH_HOURS else (input_range * step_s) // 3600
    to_dt       = now_utc
    from_dt     = to_dt - timedelta(hours=window_h)
    from_ms     = int(from_dt.timestamp() * 1000)
    to_ms       = int(to_dt.timestamp() * 1000)

    short_input = input_ref.split('/')[-1]
    short_model = object_ref.split('/')[-1] if '/' in object_ref else object_ref
    print(f'[{short_model}]  archive={short_input}  window={window_h}h  step={step_s}s', end='  ')

    entry = {
        'object_ref':  object_ref,
        'input_ref':   input_ref,
        'short_model': short_model,
        'short_input': short_input,
        'step_s':      step_s,
        'window_h':    window_h,
        'from_dt':     from_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'to_dt':       to_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'status':      'ok',
        'score':       None,
        'n_expected':  None,
        'n_missing':   None,
        'n_dup':       None,
        'n_spikes':    None,
        'n_stuck':     None,
        'n_roc':       None,
        'n_long_gaps': None,
        'n_raw_points': None,
        'error':       None,
        'pipeline_result': None,
    }

    # ── Fetch ──────────────────────────────────────────────────────────────────
    raw_payload = _fetch_scada(api_url, input_ref, from_ms, to_ms, step_s,
                                timeout=BATCH_SCADA_TIMEOUT)

    if raw_payload is None:
        entry['status'] = 'scada_unavailable'
        entry['error']  = f'SCADA did not respond: {api_url}'
        print('⚠  SCADA unavailable')
        BATCH_RESULTS.append(entry)
        continue

    raw_series = raw_payload.get(input_ref, [])
    if not raw_series:
        # Some SCADA APIs return data under the last path segment
        raw_series = next(iter(raw_payload.values()), [])
    entry['n_raw_points'] = len(raw_series)

    if not raw_series:
        entry['status'] = 'no_data'
        entry['error']  = 'SCADA returned empty series'
        print('⚠  no data')
        BATCH_RESULTS.append(entry)
        continue

    # ── Pipeline ───────────────────────────────────────────────────────────────
    try:
        req = AssessRequest(
            **{'from': entry['from_dt']},
            object_ref=input_ref,
            to=entry['to_dt'],
            step=step_s,
            allow_look_ahead=ALLOW_LOOK_AHEAD,
            z_score_window=Z_SCORE_WINDOW,
            z_score_threshold=Z_SCORE_THRESHOLD,
            stuck_window=STUCK_WINDOW,
        )
        pl  = DataQualityPipeline(req)
        res = pl.run({input_ref: raw_series})
        st  = res.metrics_scoring.tags[input_ref]

        entry['score']        = res.metrics_scoring.overall_quality_score
        entry['n_expected']   = st.total_expected_points
        entry['n_missing']    = st.missing_points_count
        entry['n_dup']        = st.duplicates_count
        entry['n_spikes']     = st.outliers_count
        entry['n_stuck']      = st.stuck_sequences_count
        entry['n_roc']        = st.rate_of_change_count
        entry['n_long_gaps']  = st.long_gaps_count
        entry['pipeline_result'] = res

        badge = '✅' if entry['score'] >= 80 else '⚠️ ' if entry['score'] >= 60 else '❌'
        print(f'{badge} score={entry["score"]:.1f}  miss={st.missing_points_count}')

    except Exception as exc:
        entry['status'] = 'pipeline_error'
        entry['error']  = str(exc)
        print(f'❌ pipeline error: {exc}')

    BATCH_RESULTS.append(entry)

print(f'\nDone: {len(BATCH_RESULTS)} inputs assessed.')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────

df_summary = pd.DataFrame([
    {
        'Model':       r['short_model'],
        'Archive':     r['short_input'],
        'Step':        f"{r['step_s']}s",
        'Window':      f"{r['window_h']}h",
        'Status':      r['status'],
        'Score':       r['score'],
        'Expected':    r['n_expected'],
        'Raw pts':     r['n_raw_points'],
        'Missing':     r['n_missing'],
        'Duplicates':  r['n_dup'],
        'Spikes':      r['n_spikes'],
        'Stuck':       r['n_stuck'],
        'RoC':         r['n_roc'],
        'Long gaps':   r['n_long_gaps'],
        'Error':       r['error'],
    }
    for r in BATCH_RESULTS
])

def _color_score(val):
    if pd.isna(val):
        return 'background-color: #f5b7b1'
    if val >= 80:
        return 'background-color: #d5f5e3'
    if val >= 60:
        return 'background-color: #fdebd0'
    return 'background-color: #f5b7b1'

print('Batch quality assessment summary:')
display(
    df_summary.style
    .applymap(_color_score, subset=['Score'])
    .format({'Score': lambda v: f'{v:.1f}' if pd.notna(v) else '—'})
    .set_properties(**{'text-align': 'right'})
)

In [ ]:
# ── Batch visualisation ───────────────────────────────────────────────────────

ok_rows = [r for r in BATCH_RESULTS if r['score'] is not None]

if not ok_rows:
    print('⚠  No successful assessments to visualise.')
else:
    # ── 7.1 Score bar chart (one bar per model/archive) ──────────────────────
    labels  = [f"{r['short_model']}\n{r['short_input']}" for r in ok_rows]
    scores  = [r['score'] for r in ok_rows]
    colors  = ['#2ecc71' if s >= 80 else '#e67e22' if s >= 60 else '#e74c3c'
               for s in scores]

    fig_bar = go.Figure(go.Bar(
        x=labels, y=scores,
        marker_color=colors,
        text=[f'{s:.1f}' for s in scores],
        textposition='outside',
    ))
    fig_bar.add_hline(y=80, line_dash='dash', line_color='#e74c3c',
                      annotation_text='threshold 80', annotation_position='top right')
    fig_bar.update_layout(
        title='Input Data Quality — All Models',
        yaxis=dict(title='Score / 100', range=[0, 115]),
        xaxis_title='Model / Archive',
        template='plotly_white', height=420,
    )
    fig_bar.show()

    # ── 7.2 Stacked defect bar (one column per model/archive) ────────────────
    defect_cols = ['Missing', 'Duplicates', 'Spikes', 'Stuck', 'RoC']
    defect_keys = ['n_missing', 'n_dup', 'n_spikes', 'n_stuck', 'n_roc']
    defect_colors = ['#3498db', '#9b59b6', '#e67e22', '#f39c12', '#1abc9c']

    fig_def = go.Figure()
    for col, key, color in zip(defect_cols, defect_keys, defect_colors):
        fig_def.add_trace(go.Bar(
            name=col,
            x=labels,
            y=[r.get(key) or 0 for r in ok_rows],
            marker_color=color,
        ))
    fig_def.update_layout(
        barmode='stack',
        title='Defect Breakdown — All Models',
        xaxis_title='Model / Archive', yaxis_title='Point count',
        template='plotly_white', height=400,
    )
    fig_def.show()

    # ── 7.3 Completeness ratio (raw vs expected) ──────────────────────────────
    completeness = [
        round(100 * (r['n_raw_points'] or 0) / r['n_expected'], 1)
        if r['n_expected'] else None
        for r in ok_rows
    ]
    fig_comp = go.Figure(go.Bar(
        x=labels,
        y=completeness,
        marker_color=['#2ecc71' if (v or 0) >= 95 else '#e67e22' if (v or 0) >= 80 else '#e74c3c'
                      for v in completeness],
        text=[f'{v}%' if v is not None else '—' for v in completeness],
        textposition='outside',
    ))
    fig_comp.add_hline(y=95, line_dash='dash', line_color='#e74c3c',
                       annotation_text='95%', annotation_position='top right')
    fig_comp.update_layout(
        title='Raw Data Completeness (received vs expected points)',
        yaxis=dict(title='%', range=[0, 115]),
        xaxis_title='Model / Archive',
        template='plotly_white', height=380,
    )
    fig_comp.show()

In [ ]:
# ── Per-archive detail: cleaned series overlay ────────────────────────────────
# One line per successfully assessed archive on a shared time axis.

ok_rows = [r for r in BATCH_RESULTS if r['pipeline_result'] is not None]

if not ok_rows:
    print('⚠  No pipeline results to plot.')
else:
    pal = ['#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c',
           '#3498db', '#e67e22', '#1a5276', '#7d3c98', '#117a65']

    fig_all = go.Figure()
    for i, r in enumerate(ok_rows):
        res   = r['pipeline_result']
        tag   = r['input_ref']
        df_cl = pd.DataFrame(res.cleaned_data)
        df_cl['ts'] = pd.to_datetime(df_cl['timestamp'], utc=True).dt.tz_convert(None)

        if tag not in df_cl.columns:
            continue

        label = f"{r['short_model']} ({r['score']:.0f}/100)"
        fig_all.add_trace(go.Scatter(
            x=df_cl['ts'],
            y=df_cl[tag].astype(float),
            mode='lines',
            name=label,
            line=dict(color=pal[i % len(pal)], width=1.6),
        ))

    fig_all.update_layout(
        title='Cleaned Input Series — All Archives',
        xaxis_title='Time (UTC)', yaxis_title='Value',
        hovermode='x unified', template='plotly_white', height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1),
    )
    fig_all.show()

In [ ]:
# ── Export batch summary ──────────────────────────────────────────────────────

export_dir = pathlib.Path('.') / 'exports'
export_dir.mkdir(exist_ok=True)
ts_now = datetime.now().strftime('%Y%m%d_%H%M%S')

# Summary CSV
fpath_summary = export_dir / f'batch_quality_summary__{ts_now}.csv'
df_summary.to_csv(fpath_summary, index=False)
print(f'✅  Summary CSV   → {fpath_summary}')

# Per-archive anomaly logs (one CSV per archive)
for r in BATCH_RESULTS:
    if r['pipeline_result'] is None:
        continue
    alog = r['pipeline_result'].anomalies_log
    if not alog:
        continue
    slug = r['short_input'][:40].replace(' ', '_')
    fpath_al = export_dir / f'batch_anomalies__{slug}__{ts_now}.csv'
    pd.DataFrame([a.model_dump() for a in alog]).to_csv(fpath_al, index=False)
    print(f'✅  Anomaly log   → {fpath_al}  ({len(alog)} records)')

# Full JSON batch report
batch_report = [
    {
        'object_ref':  r['object_ref'],
        'input_ref':   r['input_ref'],
        'from':        r['from_dt'],
        'to':          r['to_dt'],
        'step_s':      r['step_s'],
        'status':      r['status'],
        'score':       r['score'],
        'stats': {
            'n_expected':  r['n_expected'],
            'n_raw_points': r['n_raw_points'],
            'n_missing':   r['n_missing'],
            'n_dup':       r['n_dup'],
            'n_spikes':    r['n_spikes'],
            'n_stuck':     r['n_stuck'],
            'n_roc':       r['n_roc'],
            'n_long_gaps': r['n_long_gaps'],
        },
        'error': r['error'],
    }
    for r in BATCH_RESULTS
]
fpath_json = export_dir / f'batch_quality_report__{ts_now}.json'
with open(fpath_json, 'w') as f:
    json.dump(batch_report, f, indent=2, ensure_ascii=False, default=str)
print(f'✅  Batch report  → {fpath_json}')